# Laboratorio #4 - Aprendizaje por Refuerzo
* Paula Barillas - 22764
* Gerardo Pineda - 22880
* Mónica Salvatierra - 22249
* Bianca Calderón - 22272

- Link del repositorio: https://github.com/alee2602/LAB4-RL


## **Task 1**

Una empresa de logística de última milla está evaluando el uso de robots autónomos para la gestión interna de su almacén principal. El almacén se modela como una cuadrícula de 8 × 8con puntos de recogida, puntos de entrega, zonas de penalización por congestión, y obstáculos fijos. El equipo de ingeniería necesita comparar dos estrategias de aprendizaje antes de comprometer recursos en un sistema completo: una política conservadora que aprenda a navegar de forma segura durante el entrenamiento, y una política agresiva que busque la ruta óptima sin importar los riesgos durante la exploración. Su grupo ha sido contratado para implementar ambas estrategias usando SARSA y Q-Learning respectivamente, comparar su comportamiento empírico, y producir un dictamen técnico con recomendaciones concretas para la gerencia

Diseñen formalmente el MDP que representa el almacén. El diseño debe especificar:

**1. El espacio de estados y el espacio de acciones. Justifiquen cada decisión considerando la interfaz de Gymnasium: observation_space y action_space deben ser instancias de gymnasium.spaces.Discrete o gymnasium.spaces.Box según corresponda. Argumenten cuál es más apropiado para este dominio.**

En el presente contexto, lo más apropiado es gymnasium.spaces.Discrete para ambos espacios, no Box. Cada celda de la cuadrícula 8x8 se codifica como un entero único con `estado = fila * 8 + columna`, dando Discrete(64), porque el agente siempre ocupa una posición exacta y no hay noción continua de ubicación. El uso de box tendría sentido con coordenadas continuas o imágenes como observación, pero aquí solo añadiría complejidad innecesaria. Para las acciones, el agente se mueve en cuatro direcciones, lo que se representa con Discrete(4).

**2. La función de recompensa con al menos tres componentes: recompensa por entrega exitosa, penalización por zona de congestión, y penalización por paso. Justifiquen la magnitud relativa de cada componente y argumenten qué comportamiento indeseable produciría una ponderación incorrecta de alguno de ellos.**

- **Recompensa por entrega exitosa:** un valor grande y positivo, por ejemplo +70, otorgado solo al llegar al punto de entrega, para que domine claramente sobre las penalizaciones acumuladas en una trayectoria típica.

- **Penalización por zona de congestión:** un valor negativo moderado, por ejemplo -5, aplicado al estar sobre una celda de congestión, suficiente para desincentivar pero sin que las rutas alternativas sean incoherentes.

- **Penalización por paso:** un valor pequeño y negativo, por ejemplo -1, en cada transición, para incentivar el uso de rutas cortas.

En este caso, una mala calibración produce comportamientos indeseables. Si el paso pesa demasiado frente a la congestión, el agente cruzará zonas de riesgo con tal de ahorrar tiempo. Si la congestión pesa demasiado frente a la entrega, el agente empezará a dar vueltas o evitará completar la tarea. Si la entrega no domina lo suficiente, el agente puede minimizar penalizaciones sin nunca llegar a la meta.

**3. La condición de terminación del episodio: ¿cuándo termina un episodio? ¿Es apropiado tener un
límite máximo de pasos? Justifiquen.**

- El episodio termina cuando el agente llega al punto de entrega. También conviene agregar un límite máximo de pasos, por ejemplo 100, como truncamiento. Esto evita que una política casi aleatoria en las primeras fases pueda quedar atrapada en ciclos infinitos. En Gymnasium esto se distingue con `terminated` (llegó a la meta) y `truncated` (se acabó el tiempo), siendo de utilidad para posteriores análisis. 

**4. El diseño del mapa 8 × 8: ubiquen al menos dos zonas de congestión adyacentes a rutas de altarecompensa. Esta configuración específica es crítica para que la comparación entre SARSA y Q-Learning sea informativa. Expliquen por qué esa configuración espacial genera el comportamiento
diferencial esperado entre ambos algoritmos.**

Consideramos que conviene ubicar al menos dos zonas de congestión sobre o muy cerca del camino más corto entre recogida y entrega. Por ejemplo, con inicio en la esquina superior izquierda y entrega en la inferior derecha, una franja de congestión puede cruzar la diagonal natural, dejando una ruta alternativa más larga que la rodea.

Esta configuración es la que hace útil la comparación entre algoritmos. Q-Learning es off-policy y aprende el valor máximo del siguiente estado sin importar la acción exploratoria tomada, por lo que su política greedy final puede explotar el camino corto arriesgado si la recompensa neta lo justifica. SARSA es on-policy y actualiza según la acción que realmente tomará bajo su política, incluyendo exploración epsilon-greedy, lo que lo hace más sensible al riesgo de caer en congestión por una acción aleatoria y lo lleva a preferir rutas más conservadoras. Sin esta adyacencia entre congestión y ruta óptima, ambos algoritmos convergerían a políticas casi idénticas, por lo que no habría punto de comparación.

## **Task 2**

**1. Para el entorno que diseñaron, predigan formalmente cuál algoritmo, SARSA o Q-Learning, producirá
mayor recompensa acumulada durante el entrenamiento y cuál producirá mayor recompensa
durante la evaluación con política greedy pura. Justifiquen cada predicción usando las propiedades
on-policy y off-policy de cada algoritmo y la estructura específica de su mapa**

Q-Learning (off-policy) actualiza con $max_{a'} Q(s',a')$, ignorando que la exploración ε-greedy real puede empujar al agente hacia la congestión. Esto le hace sobrevalorar el camino corto y arriesgado. SARSA (on-policy) actualiza con Q(s',a') donde a' sí viene de la política ε-greedy real, así que "siente" el riesgo de caer en congestión por una acción exploratoria y aprende a preferir la ruta larga y segura.

En entrenamiento, ambos exploran con ε>0: Q-Learning cae más veces en congestión porque su política greedy está pegada al borde y SARSA acumula más recompensa durante el entrenamiento. En evaluación greedy ε=0: Q-Learning ya convergió a Q* (el óptimo real, sin castigo por exploración) y toma el atajo sin pagar el costo. SARSA quedó sesgado hacia la ruta conservadora que aprendió, aunque ya no sea necesaria. Por lo tanto Q-Learning gana en evaluación. En general, SARSA gana en entrenamiento y Q-Learning gana en evaluación greedy pura.


**2. Argumenten cómo afecta el valor de εal comportamiento diferencial entre SARSA y Q-Learning en su
entorno. ¿Existe un valor de εpara el cual ambos algoritmos convergen a políticas idénticas?
Justifiquen matemáticamente**

El parámetro ε controla la magnitud de la brecha entre ambos algoritmos, porque es precisamente el término que aparece en el target de SARSA y no en el de Q-Learning. Como $Σ_{a'} π_ε(a'|s')·Q(s',a') = (1−ε)·max_{a'}Q(s',a') + (ε/|A|)·Σ_{a'}Q(s',a')$, la diferencia entre ambos targets es proporcional a ε: mientras mayor sea ε, mayor peso reciben las acciones subóptimas (incluyendo caer en congestión), y mayor será el sesgo conservador de SARSA respecto a Q-Learning. Con ε bajo, ambos targets se parecen más y la brecha entre las políticas se reduce.

Sí, en el límite ε→0. Cuando ε->0, $π_ε(a'|s') -> 𝟙[a'=argmax]$, por lo que: $Σ_{a'} π_ε(a'|s')·Q(s',a') -> max_{a'} Q(s',a')$. Las dos ecuaciones de punto fijo coinciden, dando Q_SARSA -> Q*. Esto es consistente con la teoría de convergencia de Greedy in the Limit with Infinite Exploration, si ε se decae a 0 con una tasa apropiada durante el entrenamiento, manteniendo exploración infinita en el límite temprano, tanto SARSA como Q-Learning convergen a la misma política óptima determinística. Cabe notar que con ε estrictamente igual a 0 desde el inicio no habría exploración y el aprendizaje podría quedar atrapado en óptimos locales sin garantía de cobertura completa del espacio estado-acción; por eso la convergencia formal se enuncia como un límite: $ε_t → 0$, no como un valor fijo utilizable en la práctica.


**3. Para su función de recompensa específica, calculen una cota superior del valor óptimo del
estado inicial, asumiendo que el agente siempre toma la ruta más corta sin pasar por zonas de
congestión. Usen esta cota como referencia para evaluar qué tan cerca llegan sus implementaciones
al óptimo teórico**

Teniendo en cuenta que grid 8×8 con inicio en (0,0) y entrega en (7,7), recompensa de entrega +70, penalización por paso −1, γ como factor de descuento, tomamos γ=1 por ser un entorno episódico de horizonte finito, típico en estas tareas. la distancia Manhattan entre (0,0) y (7,7) es d = |7−0| + |7−0| = 14 pasos, lograble moviéndose solo en dirección derecha/abajo, sin desperdiciar ningún paso. Esta es la ruta más corta posible en la cuadrícula, independientemente de si cruza o no congestión.

Con Cota superior γ=1: $V*(s₀) ≤ 70 + 14·(−1) = 70 − 14 = 56$ y con γ descontando cada paso: $V*(s₀) ≤ −Σ_{t=0}^{d−1} γ^t + 70·γ^d = −(1−γ^d)/(1−γ) + 70·γ^d$

Esta cifra asume la mejor situación posible, el camino más corto matemáticamente permitido por la geometría de la cuadrícula, sin ningún paso adicional y sin pisar ninguna celda de congestión. Dado que en nuestro diseño colocamos deliberadamente zonas de congestión sobre o cerca de la diagonal natural entre inicio y entrega, es muy probable que la ruta realmente óptima, la que maximiza recompensa esperada considerando la penalización de congestión, deba desviarse de esa diagonal, incrementando su longitud real a d_real > 14. Por lo tanto, el valor óptimo verdadero cumplirá: $V*(s₀) ≤ 56$. Esta cota sirve como referencia, si al evaluar las políticas entrenadas obtenemos recompensas cercanas a 56, sabremos que el agente encontró o casi el atajo óptimo ignorando el riesgo de congestión, en cambio si obtenemos valores notablemente menores, eso refleja el costo que cada algoritmo paga por evitar o tomar el riesgo de la zona de congestión.


## **Task 3**

In [1]:
import numpy as np
import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces

In [2]:
class WarehouseEnv(gym.Env):
    metadata = {"render_modes": ["human"]}

    def __init__(self, size=8, max_steps=100):
        super().__init__()
        self.size = size
        self.max_steps = max_steps

        # observation_space y action_space como Discrete
        self.observation_space = spaces.Discrete(size * size)
        self.action_space = spaces.Discrete(4)

        self.start = (0, 0)
        self.goal = (size - 1, size - 1)

        # celdas intransitables
        self.obstacles = {(1, 5), (6, 1), (4, 6)}
        
        self.congestion = {
            (2, 2), (2, 3), (3, 3), (3, 4),
            (4, 4), (4, 5), (5, 5), (5, 6)
        }

        # deltas de movimiento por acción
        self._action_to_delta = {
            0: (-1, 0),  # arriba
            1: (1, 0),   # abajo
            2: (0, -1),  # izquierda
            3: (0, 1),   # derecha
        }

        self.max_steps = max_steps
        self._step_count = 0
        self._agent_pos = self.start

    def _pos_to_state(self, pos):
        row, col = pos
        return row * self.size + col

    def reset(self, seed=None, options=None):
        super().reset(seed=seed)
        self._agent_pos = self.start
        self._step_count = 0
        info = {"congestion": False}
        return self._pos_to_state(self._agent_pos), info

    def step(self, action):
        row, col = self._agent_pos
        drow, dcol = self._action_to_delta[action]
        new_row = np.clip(row + drow, 0, self.size - 1)
        new_col = np.clip(col + dcol, 0, self.size - 1)
        new_pos = (new_row, new_col)

        # si la celda destino es un obstáculo, el agente no se mueve
        if new_pos in self.obstacles:
            new_pos = (row, col)

        self._agent_pos = new_pos
        self._step_count += 1

        # penalización por paso (-1)
        reward = -1.0
        in_congestion = self._agent_pos in self.congestion
        if in_congestion:
            reward += -5.0 

        terminated = self._agent_pos == self.goal
        if terminated:
            reward += 70.0 

        truncated = self._step_count >= self.max_steps

        info = {"congestion": in_congestion}
        return self._pos_to_state(self._agent_pos), reward, terminated, truncated, info

    def render(self):
        grid = np.full((self.size, self.size), ".", dtype=str)
        for (r, c) in self.obstacles:
            grid[r, c] = "#"
        for (r, c) in self.congestion:
            grid[r, c] = "x"
        gr, gc = self.goal
        grid[gr, gc] = "G"
        ar, ac = self._agent_pos
        grid[ar, ac] = "A"
        print("\n".join(" ".join(row) for row in grid))
        print()

In [3]:
def epsilon_greedy(Q, state, epsilon, n_actions, rng):
    # política de comportamiento e-greedy
    if rng.random() < epsilon:
        return rng.integers(n_actions)
    return int(np.argmax(Q[state]))


def sarsa(env, num_episodes=2000, alpha=0.1, gamma=1.0, epsilon=0.1, seed=42):
    rng = np.random.default_rng(seed)
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))

    rewards_per_episode = []
    steps_per_episode = []
    congestion_per_episode = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        action = epsilon_greedy(Q, state, epsilon, n_actions, rng)

        total_reward = 0.0
        steps = 0
        congestion_count = 0
        terminated = truncated = False

        while not (terminated or truncated):
            next_state, reward, terminated, truncated, info = env.step(action)
            next_action = epsilon_greedy(Q, next_state, epsilon, n_actions, rng)
            # Q(S_t,A_t) <- Q(S_t,A_t) + alpha*[R_{t+1} + gamma*Q(S_{t+1},A_{t+1}) - Q(S_t,A_t)]
            td_target = reward + gamma * Q[next_state, next_action] * (not terminated)
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            state, action = next_state, next_action
            total_reward += reward
            steps += 1
            if info["congestion"]:
                congestion_count += 1

        rewards_per_episode.append(total_reward)
        steps_per_episode.append(steps)
        congestion_per_episode.append(congestion_count)

    return Q, rewards_per_episode, steps_per_episode, congestion_per_episode

In [4]:
def q_learning(env, num_episodes=2000, alpha=0.1, gamma=1.0, epsilon=0.1, seed=42):
    rng = np.random.default_rng(seed)
    n_states = env.observation_space.n
    n_actions = env.action_space.n
    Q = np.zeros((n_states, n_actions))

    rewards_per_episode = []
    steps_per_episode = []
    congestion_per_episode = []

    for ep in range(num_episodes):
        state, _ = env.reset()

        total_reward = 0.0
        steps = 0
        congestion_count = 0
        terminated = truncated = False

        while not (terminated or truncated):
            action = epsilon_greedy(Q, state, epsilon, n_actions, rng)
            next_state, reward, terminated, truncated, info = env.step(action)
            # Q(S_t,A_t) <- Q(S_t,A_t) + alpha*[R_{t+1} + gamma*max_a' Q(S_{t+1},a') - Q(S_t,A_t)]
            best_next = np.max(Q[next_state]) if not terminated else 0.0
            td_target = reward + gamma * best_next
            td_error = td_target - Q[state, action]
            Q[state, action] += alpha * td_error

            state = next_state
            total_reward += reward
            steps += 1
            if info["congestion"]:
                congestion_count += 1

        rewards_per_episode.append(total_reward)
        steps_per_episode.append(steps)
        congestion_per_episode.append(congestion_count)

    return Q, rewards_per_episode, steps_per_episode, congestion_per_episode

In [5]:
def evaluate_policy(env, Q, num_episodes=100, seed=123):
    rng = np.random.default_rng(seed)
    rewards_per_episode = []
    steps_per_episode = []
    congestion_per_episode = []

    for ep in range(num_episodes):
        state, _ = env.reset()
        total_reward = 0.0
        steps = 0
        congestion_count = 0
        terminated = truncated = False

        while not (terminated or truncated):
            action = int(np.argmax(Q[state]))  # epsilon = 0
            state, reward, terminated, truncated, info = env.step(action)
            total_reward += reward
            steps += 1
            if info["congestion"]:
                congestion_count += 1

        rewards_per_episode.append(total_reward)
        steps_per_episode.append(steps)
        congestion_per_episode.append(congestion_count)

    return rewards_per_episode, steps_per_episode, congestion_per_episode

In [6]:
env = WarehouseEnv(size=8, max_steps=100)

Q_sarsa, r_sarsa_train, s_sarsa_train, c_sarsa_train = sarsa(
    env, num_episodes=2000, alpha=0.1, gamma=1.0, epsilon=0.1
)
Q_qlearning, r_ql_train, s_ql_train, c_ql_train = q_learning(
    env, num_episodes=2000, alpha=0.1, gamma=1.0, epsilon=0.1
)

r_sarsa_eval, s_sarsa_eval, c_sarsa_eval = evaluate_policy(env, Q_sarsa, num_episodes=100)
r_ql_eval, s_ql_eval, c_ql_eval = evaluate_policy(env, Q_qlearning, num_episodes=100)

print("SARSA entrenamiento: recompensa promedio =", np.mean(r_sarsa_train[-100:]))
print("SARSA evaluación greedy: recompensa promedio =", np.mean(r_sarsa_eval),
      "pasos promedio =", np.mean(s_sarsa_eval),
      "congestión promedio =", np.mean(c_sarsa_eval))

print("Q-Learn entrenamiento: recompensa promedio =", np.mean(r_ql_train[-100:]))
print("Q-Learn evaluación greedy: recompensa promedio =", np.mean(r_ql_eval),
      "pasos promedio =", np.mean(s_ql_eval),
      "congestión promedio =", np.mean(c_ql_eval))

SARSA entrenamiento: recompensa promedio = 54.22
SARSA evaluación greedy: recompensa promedio = 56.0 pasos promedio = 14.0 congestión promedio = 0.0
Q-Learn entrenamiento: recompensa promedio = 54.26
Q-Learn evaluación greedy: recompensa promedio = 56.0 pasos promedio = 14.0 congestión promedio = 0.0


In [7]:
env.reset()
env.render()

A . . . . . . .
. . . . . # . .
. . x x . . . .
. . . x x . . .
. . . . x x # .
. . . . . x x .
. # . . . . . .
. . . . . . . G



## Uso de IA generativa — Task 3

**Prompt utilizado:**

“Ayúdameeee, necesito hacer la clase WarehouseEnv usando gymnasium.Env . El entorno debe seguir el MDP que hicimos en el Task 1, con una cuadrícula de 8x8, recompensas de +70, -5 y -1, y zonas de congestión cerca de la diagonal.  En donde también necesito guardar las recompensas, pasos y visitas a las zonas de congestión de cada episodio.”

**Por qué funcionó este prompt:**

Funcionó porque incluimos los datos del MDP que ya habíamos definido en el Task 1. Así, la implementación siguió el mismo diseño y no se tuvo que crear un entorno diferente. Después revisamos el código y ajustamos algunas partes para que las actualizaciones de SARSA y Q-Learning coincidieran.
